# ETL & Data Preprocessing

**Цель:** воспроизводимая подготовка транзакционных данных для EDA, ABC/XYZ и RFM.

Ноутбук не удаляет строки из-за отсутствия необязательных атрибутов (например, возраста или рейтинга). Бизнес-правило реализованной продажи зафиксировано явно и вынесено в конфигурацию.

## Уточнение по словарю данных

Словарь определяет `Sales_Amount` как **Recorded net sales amount**, а `Profit` как **Recorded transaction profit**. Поэтому эти суммы используются из источника без повторного применения скидки и без подмены `Profit` формулой `Sales_Amount - Cost_Amount`.

`Order_Status` описан как операционный статус, а `Return_Flag` — как индикатор возврата. Словарь **не задаёт** официальное правило признания выручки. Для воспроизводимости в проекте используется явное аналитическое допущение: в реализованные продажи входят строки со статусом `Completed` и `Return_Flag != Yes`.


### Контроль ключей заказов

В исходном CSV `Order_ID` встречается у нескольких разных `Customer_ID`, поэтому он не является глобально уникальным в наблюдаемом наборе. Для RFM Frequency и заказных агрегатов используется составной ключ `(Customer_ID, Order_ID)`. `Transaction_ID` после удаления 45 полных дубликатов уникален.


## Этап 1. Загрузка и аудит

In [10]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Пути определяются относительно расположения ноутбука / текущей рабочей директории.
CWD = Path.cwd().resolve()
CANDIDATES = [CWD, *CWD.parents, Path('/mnt/data')]
BASE_DIR = next((p for p in CANDIDATES if (p / 'Sales_transactions_2022_2025.csv').exists()), CWD)
RAW_PATH = BASE_DIR / 'Sales_transactions_2022_2025.csv'
PROCESSED_DIR = BASE_DIR / 'processed'
RAW_OUTPUT_DIR = BASE_DIR / 'raw'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Словарь полей: поддерживаем каноническое имя и имя загруженного файла.
DICT_CANDIDATES = [
    BASE_DIR / 'sales_data_dictionary.csv',
    BASE_DIR / 'sales_data_dictionary(1).csv',
]
SALES_DICT_PATH = next((p for p in DICT_CANDIDATES if p.exists()), None)

assert RAW_PATH.exists(), f'Не найден исходный CSV: {RAW_PATH}'
transactions = pd.read_csv(RAW_PATH)
transactions_copy = transactions.copy()
transactions_copy.columns = transactions_copy.columns.str.strip().str.lower()

print(f'Файл: {RAW_PATH}')
print(f'Размер исходных данных: {transactions_copy.shape[0]:,} строк × {transactions_copy.shape[1]} столбцов')
print(f'Полные дубликаты: {transactions_copy.duplicated().sum():,}')
print(f'Дубли Transaction_ID: {transactions_copy.duplicated("transaction_id").sum():,}')
print(f'Пропуски по столбцам:\n{transactions_copy.isna().sum().sort_values(ascending=False).head(12)}')
if SALES_DICT_PATH is not None:
    sales_dict = pd.read_csv(SALES_DICT_PATH)
    required_dictionary_columns = {'Column_Name', 'Description', 'Data_Type', 'Business_Meaning'}
    missing_dictionary_columns = required_dictionary_columns - set(sales_dict.columns)
    if missing_dictionary_columns:
        raise ValueError(f'В словаре отсутствуют колонки: {sorted(missing_dictionary_columns)}')
    dictionary_fields = set(sales_dict['Column_Name'].astype(str).str.strip())
    print(f'Словарь данных загружен: {sales_dict.shape}; файл: {SALES_DICT_PATH.name}')
    print('Ключевые определения: Sales_Amount = recorded net sales amount; Profit = recorded transaction profit;')
    print('Order_Status = operational status; Return_Flag = return indicator.')
else:
    sales_dict = None
    print('ВНИМАНИЕ: словарь не найден; бизнес-семантика полей должна быть подтверждена отдельно.')


Файл: /Users/caseynger/DE/final_work_fa/Sales_transactions_2022_2025.csv
Размер исходных данных: 18,045 строк × 36 столбцов
Полные дубликаты: 45
Дубли Transaction_ID: 45
Пропуски по столбцам:
return_reason      16168
promotion_code      9737
customer_rating     3435
customer_age         218
inventory_level      198
customer_gender      162
delivery_days        162
payment_method       147
product_name         144
customer_name        108
transaction_id         0
sales_amount           0
dtype: int64
Словарь данных загружен: (36, 5); файл: sales_data_dictionary.csv
Ключевые определения: Sales_Amount = recorded net sales amount; Profit = recorded transaction profit;
Order_Status = operational status; Return_Flag = return indicator.


## Этап 2. Очистка дубликатов и нормализация

Удаляются только полные дубликаты. Повторяющиеся `transaction_id` после этого проверяются отдельно: нельзя без бизнес-правила автоматически удалять строки, если остальные поля отличаются.

In [11]:
# Удаляем только полностью идентичные строки.
rows_before = len(transactions_copy)
transactions_copy = transactions_copy.drop_duplicates().copy()
print(f'Удалено полных дубликатов: {rows_before - len(transactions_copy):,}')

# Нормализация строковых полей: trim; сохраняем исходный регистр содержимого,
# затем точечно нормализуем категориальные поля.
for col in transactions_copy.select_dtypes(include='object').columns:
    transactions_copy[col] = transactions_copy[col].str.strip()

# Типы данных
transactions_copy['order_date'] = pd.to_datetime(transactions_copy['order_date'], errors='coerce')
transactions_copy['order_time'] = pd.to_datetime(
    transactions_copy['order_time'], format='%H:%M:%S', errors='coerce'
).dt.time
transactions_copy['customer_age'] = transactions_copy['customer_age'].fillna(0).astype('Int64')

numeric_cols = ['customer_age','quantity','unit_price','discount_percentage','sales_amount',
                'cost_amount','profit','delivery_days','customer_rating','inventory_level','order_year']
for col in numeric_cols:
    transactions_copy[col] = pd.to_numeric(transactions_copy[col], errors='coerce')

# Категории: единый регистр/написание без изменения бизнес-смысла.
for col in ['order_status','return_flag','sales_channel','shipping_method','payment_method',
            'customer_gender','customer_segment','product_category','product_subcategory']:
    if col in transactions_copy:
        transactions_copy[col] = transactions_copy[col].astype('string').str.strip()

transactions_copy['city'] = transactions_copy['city'].str.lower()
transactions_copy['customer_gender'] = (transactions_copy['customer_gender']
                                 .str.lower()
                                 .replace({'non-binary': 'non binary',})
                                 .fillna('unknown'))
transactions_copy['product_category'] = transactions_copy['product_category'].str.lower()
transactions_copy['order_status'] = transactions_copy['order_status'].str.lower().str.replace(r'\s+', ' ', regex=True)
transactions_copy['return_flag'] = transactions_copy['return_flag'].str.lower()
transactions_copy['sales_channel'] = transactions_copy['sales_channel'].str.strip()
transactions_copy['payment_method'] = transactions_copy['payment_method'].str.lower().replace({'debitcard': 'debit card'})
transactions_copy['payment_method'] = transactions_copy['payment_method'].fillna('unknown')
transactions_copy['return_reason'] = transactions_copy['return_reason'].fillna('No_Return')
transactions_copy['has_promotion'] = transactions_copy['promotion_code'].notna().astype('int8')
transactions_copy['promotion_code'] = transactions_copy['promotion_code'].fillna('NO_PROMO')

# Audit key uniqueness: retain rows, report collisions for investigation.
transaction_id_duplicates = transactions_copy[transactions_copy.duplicated('transaction_id', keep=False)].sort_values('transaction_id')
print(f'Строк с повторяющимся transaction_id: {len(transaction_id_duplicates):,}')
print('Статусы:', transactions_copy['order_status'].value_counts(dropna=False).to_dict())
print('Каналы:', transactions_copy['sales_channel'].value_counts(dropna=False).to_dict())


Удалено полных дубликатов: 45
Строк с повторяющимся transaction_id: 0
Статусы: {'completed': 11536, 'shipped': 2554, 'returned': 1597, 'processing': 1420, 'cancelled': 893}
Каналы: {'Online': 8195, 'Retail Store': 6221, 'B2B Portal': 2124, 'Phone Order': 1460}


/var/folders/_4/tbmyt5yn13q6xy17_v326ghm0000gn/T/ipykernel_31832/2432794037.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in transactions_copy.select_dtypes(include='object').columns:


### Преобразование типов данных и контроль качества

In [3]:
quality_checks = pd.Series({
    'missing_order_date': transactions_copy['order_date'].isna().sum(),
    'missing_customer_id': transactions_copy['customer_id'].isna().sum(),
    'missing_product_id': transactions_copy['product_id'].isna().sum(),
    'quantity_le_zero': (transactions_copy['quantity'] <= 0).sum(),
    'negative_unit_price': (transactions_copy['unit_price'] < 0).sum(),
    'discount_out_of_range': ((transactions_copy['discount_percentage'] < 0) | (transactions_copy['discount_percentage'] > 100)).sum(),
    'negative_sales_amount': (transactions_copy['sales_amount'] < 0).sum(),
    'negative_profit': (transactions_copy['profit'] < 0).sum(),
})
print(quality_checks.to_string())

# Сверка финансовых формул — диагностическая, не переписываем исходные суммы.
expected_sales = transactions_copy['quantity'] * transactions_copy['unit_price'] * (1 - transactions_copy['discount_percentage'].fillna(0) / 100)
sales_diff = (transactions_copy['sales_amount'] - expected_sales).abs()
profit_diff = (transactions_copy['profit'] - (transactions_copy['sales_amount'] - transactions_copy['cost_amount'])).abs()
print(f'\nSales_Amount vs qty*price*(1-discount): mismatches > $0.02 = {(sales_diff > 0.02).sum():,}')
print(f'Profit vs sales-cost: mismatches > $0.02 = {(profit_diff > 0.02).sum():,}')
print('Суммы не пересчитываются автоматически: возможны налоги, корректировки, округления или иные правила источника.')


missing_order_date         0
missing_customer_id        0
missing_product_id         0
quantity_le_zero           8
negative_unit_price        0
discount_out_of_range      1
negative_sales_amount      0
negative_profit          231

Sales_Amount vs qty*price*(1-discount): mismatches > $0.02 = 616
Profit vs sales-cost: mismatches > $0.02 = 110
Суммы не пересчитываются автоматически: возможны налоги, корректировки, округления или иные правила источника.


## Этап 3. Обработка аномалий

Строки с `quantity <= 0` сохраняются в отдельном файле для расследования и исключаются из набора для количественного анализа. Отрицательная прибыль не удаляется: она может быть реальным финансовым результатом.

In [4]:
ANOMALIES_PATH = RAW_OUTPUT_DIR / 'quantity_anomalies.csv'
invalid_quantity = transactions_copy.loc[transactions_copy['quantity'].isna() | (transactions_copy['quantity'] <= 0)].copy()
invalid_quantity.to_csv(ANOMALIES_PATH, index=False, encoding='utf-8-sig')

# Для основного транзакционного анализа требуем положительное количество,
# дату и ключи сущностей. Не фильтруем по customer_age / rating.
valid_mask = (
    transactions_copy['quantity'].gt(0)
    & transactions_copy['order_date'].notna()
    & transactions_copy['customer_id'].notna()
    & transactions_copy['product_id'].notna()
)
excluded_for_core = transactions_copy.loc[~valid_mask].copy()
df = transactions_copy.loc[valid_mask].copy()
print(f'Аномалии quantity сохранены: {len(invalid_quantity):,} → {ANOMALIES_PATH}')
print(f'Строк исключено из аналитического набора по обязательным полям: {len(excluded_for_core):,}')
print(f'Строк в аналитическом наборе: {len(df):,}')


Аномалии quantity сохранены: 8 → /Users/caseynger/DE/final_work_fa/raw/quantity_anomalies.csv
Строк исключено из аналитического набора по обязательным полям: 8
Строк в аналитическом наборе: 17,992


## Этап 4. Обработка пропусков

Пропуски не заполняются медианой там, где это создаёт искусственные наблюдения. Возраст и рейтинг остаются `NaN`; отсутствие промокода и причины возврата кодируется отдельно.

In [5]:
# Discount: NaN трактуется как 0 только как аналитическое допущение; исходная колонка сохранена в raw.
df['discount_percentage'] = df['discount_percentage'].fillna(0)

# Пропуск delivery_days: самовывозу можно присвоить 0 дней; прочие пропуски сохраняются.
df['delivery_days_missing_original'] = df['delivery_days'].isna().astype('int8')
df.loc[df['shipping_method'].str.lower().eq('pickup') & df['delivery_days'].isna(), 'delivery_days'] = 0

# Явные индикаторы отсутствующих необязательных атрибутов.
df['customer_age_missing'] = df['customer_age'].isna().astype('int8')
df['customer_rating_missing'] = df['customer_rating'].isna().astype('int8')


## Этап 5. Флаги аномалий и бизнес-правила

**Ключевое допущение:** реализованная продажа определяется как заказ со статусом `completed` и `return_flag = no`. `Shipped` и `Processing` не считаются завершённой реализацией. Возвращённые и отменённые заказы исключаются из net-метрик. Это правило требует подтверждения словарём/владельцем данных.

In [6]:
def mark_outliers_iqr(series: pd.Series, multiplier: float = 1.5) -> pd.Series:
    """Флаг IQR; не меняет значения, NaN не маркируется как выброс."""
    clean = pd.to_numeric(series, errors='coerce')
    q1, q3 = clean.quantile([0.25, 0.75])
    iqr = q3 - q1
    if pd.isna(iqr):
        return pd.Series(0, index=series.index, dtype='int8')
    return ((clean < q1 - multiplier * iqr) | (clean > q3 + multiplier * iqr)).fillna(False).astype('int8')

df['is_outlier_sales'] = mark_outliers_iqr(df['sales_amount'])
df['is_outlier_profit'] = mark_outliers_iqr(df['profit'])
df['is_outlier_delivery'] = mark_outliers_iqr(df['delivery_days'])
df['is_loss'] = df['profit'].lt(0).fillna(False).astype('int8')

# Фактический B2B-канал и крупная партия — два разных понятия.
df['is_b2b_channel'] = df['sales_channel'].str.lower().eq('b2b portal').astype('int8')
bulk_threshold = df['quantity'].quantile(0.95)
df['is_bulk_order_line'] = df['quantity'].ge(bulk_threshold).astype('int8')

# Флаги статуса/возврата
# Return_Flag является явным полем источника; status=returned также учитываем как возврат.
df['is_cancelled'] = df['order_status'].eq('cancelled').astype('int8')
df['is_returned'] = (df['return_flag'].eq('yes') | df['order_status'].eq('returned')).astype('int8')
df['is_completed'] = df['order_status'].eq('completed').astype('int8')

# Аналитическое допущение (не прямое правило из словаря):
# в realized sales включаем только Completed и не возвращённые строки.
# Словарь описывает Order_Status как operational status, но не задаёт момент признания выручки.
REALIZED_STATUSES = {'completed'}
df['is_realized'] = (
    df['order_status'].isin(REALIZED_STATUSES)
    & df['is_returned'].eq(0)
).astype('int8')

# Sales_Amount уже описан в словаре как recorded net sales amount:
# не пересчитываем его повторно из Quantity * Unit_Price * Discount.
# Profit также берём как recorded transaction profit, не заменяем на Sales_Amount - Cost_Amount.
df['recorded_sales_amount'] = df['sales_amount']
df['net_sales'] = df['sales_amount'].where(df['is_realized'].eq(1), 0.0)
df['net_profit'] = df['profit'].where(df['is_realized'].eq(1), 0.0)

print('Статусы:', df['order_status'].value_counts(dropna=False).to_dict())
print(f'Завершённые реализованные строки: {df.is_realized.sum():,}')
print(f'Recorded Sales_Amount (all eligible rows): {df.recorded_sales_amount.sum():,.2f}')
print(f'Net sales: {df.net_sales.sum():,.2f}')
print(f'Net profit: {df.net_profit.sum():,.2f}')
print(f'B2B Portal lines: {df.is_b2b_channel.sum():,}; bulk lines (qty ≥ {bulk_threshold:g}): {df.is_bulk_order_line.sum():,}')


Статусы: {'completed': 11531, 'shipped': 2552, 'returned': 1597, 'processing': 1419, 'cancelled': 893}
Завершённые реализованные строки: 11,349
Recorded Sales_Amount (all eligible rows): 8,229,473.04
Net sales: 5,175,010.17
Net profit: 1,040,355.97
B2B Portal lines: 2,123; bulk lines (qty ≥ 6): 925


## Этап 6. Feature Engineering

In [12]:
# Временные признаки
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter
df['order_dow'] = df['order_date'].dt.dayofweek
df['is_weekend'] = df['order_dow'].ge(5).astype('int8')
df['order_hour'] = df['order_time'].apply(lambda x: x.hour if pd.notna(x) else np.nan)
df['is_holiday_season'] = (df['order_month'].isin([11, 12]) & df['order_date'].dt.day.ge(20)).astype('int8')
df['order_month_period'] = df['order_date'].dt.to_period('M').astype(str)

# Возраст: не удаляем пропуски и не присваиваем неизвестным искусственный возраст.
df['is_age_anomaly'] = (df['customer_age'].lt(18) | df['customer_age'].gt(100)).fillna(False).astype('int8')
df['age_group'] = pd.cut(
    df['customer_age'], bins=[17, 25, 35, 45, 55, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-100'], include_lowest=True
).astype('string').fillna('Unknown')

# Маржа: отношение прибыли к продажам; при нулевой/отрицательной выручке оставляем NaN.
df['profit_margin'] = np.where(df['sales_amount'] > 0, df['profit'] / df['sales_amount'], np.nan)
df['net_profit_margin'] = np.where(df['net_sales'] > 0, df['net_profit'] / df['net_sales'], np.nan)

# Первая реализованная покупка, а не первая произвольная транзакция.
realized_dates = df.loc[df['is_realized'].eq(1)].groupby('customer_id')['order_date'].min()
df['first_realized_purchase'] = df['customer_id'].map(realized_dates)
df['is_first_purchase'] = (df['is_realized'].eq(1) & df['order_date'].eq(df['first_realized_purchase'])).astype('int8')
df['customer_order_key'] = df['customer_id'].astype('string') + '|' + df['order_id'].astype('string')

print('Feature Engineering завершён.')
print(f'Строк: {len(df):,}; клиентов: {df.customer_id.nunique():,}; SKU: {df.product_id.nunique():,}')


Feature Engineering завершён.
Строк: 17,992; клиентов: 4,882; SKU: 28


/var/folders/_4/tbmyt5yn13q6xy17_v326ghm0000gn/T/ipykernel_31832/3843439794.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['order_dow'] = df['order_date'].dt.dayofweek


## Этап 7. RFM-анализ

RFM строится **только по реализованным покупкам**. Recency — дни с последней завершённой покупки; Frequency — уникальные `order_id`; Monetary — сумма `net_sales`. Для одинаковых значений используется ранжирование с детерминированным разрешением ties, чтобы гарантировать пять скоринговых уровней при достаточном числе клиентов.

In [13]:
rfm_source = df.loc[df['is_realized'].eq(1)].copy()
if rfm_source.empty:
    raise ValueError('Нет реализованных покупок: RFM невозможно рассчитать.')

max_purchase_date = rfm_source['order_date'].max()
REFERENCE_DATE = max_purchase_date.normalize() + pd.Timedelta(days=1)

# Order_ID повторяется у разных Customer_ID; заказ идентифицируем составным ключом.
# Сначала агрегируем строки до (customer_id, order_id), затем считаем клиентские метрики.
realized_order_totals = (rfm_source.groupby(['customer_id','order_id'])
                         .agg(order_net_sales=('net_sales','sum'), order_date=('order_date','max'))
                         .reset_index())
rfm = (realized_order_totals.groupby('customer_id')
       .agg(
           recency=('order_date', lambda s: (REFERENCE_DATE - s.max().normalize()).days),
           frequency=('order_id', 'nunique'),
           monetary=('order_net_sales', 'sum'),
           avg_order_value=('order_net_sales', 'mean'),
           first_purchase=('order_date', 'min'),
           last_purchase=('order_date', 'max'),
           realized_orders=('order_id', 'nunique')
       ).reset_index())

# Детерминированный квинтильный score 1..5; ties ранжируются по стабильному порядку строк.
def quintile_score(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    # Average ranks keep equal metric values in the same score bucket.
    pct = series.rank(method='average', pct=True, ascending=True)
    score = np.ceil(pct * 5).clip(1, 5).astype('int8')
    return score if higher_is_better else (6 - score).astype('int8')

rfm['recency_score'] = quintile_score(rfm['recency'], higher_is_better=False)
rfm['frequency_score'] = quintile_score(rfm['frequency'], higher_is_better=True)
rfm['monetary_score'] = quintile_score(rfm['monetary'], higher_is_better=True)
rfm['rfm_score'] = (rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str) + rfm['monetary_score'].astype(str))

# Взаимоисключающая сегментация; условия учитывают R, F и M.
def rfm_segment(row):
    R, F, M = row['recency_score'], row['frequency_score'], row['monetary_score']
    if R >= 4 and F >= 4 and M >= 4:
        return 'Champions'
    if R >= 3 and F >= 3:
        return 'Loyal'
    if R <= 2 and M >= 4:
        return 'Lost VIPs'
    if R <= 2 and F >= 3:
        return 'At Risk'
    if R <= 2 and F <= 2:
        return 'Hibernating'
    if R >= 4 and F <= 2:
        return 'New'
    if M >= 4 and F <= 2:
        return 'Big Spenders'  # крупные по сумме и редкие, но не обязательно ушедшие
    return 'Other'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)
print(f'Reference date: {REFERENCE_DATE.date()}')
print(f'RFM customers: {len(rfm):,}')
print(rfm['segment'].value_counts(dropna=False).to_string())


Reference date: 2026-01-01
RFM customers: 4,243
segment
Loyal           1231
Hibernating      866
Champions        772
At Risk          427
Lost VIPs        406
New              299
Other            209
Big Spenders      33


## Этап 8. Сохранение результата и контроль воспроизводимости

In [9]:
SALES_CLEAN_PATH = PROCESSED_DIR / 'sales_clean.csv'
SALES_BI_READY_PATH = PROCESSED_DIR / 'sales_bi_ready.csv'
RFM_BI_READY_PATH = PROCESSED_DIR / 'rfm_bi_ready.csv'
AUDIT_PATH = PROCESSED_DIR / 'etl_audit.json'

# Оба sales-файла намеренно совпадают по схеме; sales_clean оставлен для совместимости.
df.to_csv(SALES_CLEAN_PATH, index=False, encoding='utf-8-sig')
df.to_csv(SALES_BI_READY_PATH, index=False, encoding='utf-8-sig')
rfm.to_csv(RFM_BI_READY_PATH, index=False, encoding='utf-8-sig')

order_customer_counts = df.groupby('order_id')['customer_id'].nunique()
composite_order_count = int(df[['customer_id','order_id']].drop_duplicates().shape[0])

audit = {
    'unique_order_ids_reused_across_customers': int(order_customer_counts.gt(1).sum()),
    'composite_customer_order_count': composite_order_count,
    'source_file': RAW_PATH.name,
    'source_rows': int(len(transactions)),
    'rows_after_exact_deduplication': int(len(transactions_copy)),
    'analysis_rows': int(len(df)),
    'exact_duplicates_removed': int(rows_before - len(transactions_copy)),
    'invalid_quantity_rows': int(len(invalid_quantity)),
    'analysis_excluded_rows': int(len(excluded_for_core)),
    'customers_in_rfm': int(rfm['customer_id'].nunique()),
    'date_min': str(df['order_date'].min().date()),
    'date_max': str(df['order_date'].max().date()),
    'realized_rule': "order_status == 'completed' AND (return_flag != 'yes') AND order_status != 'cancelled'",
    'realized_rows': int(df['is_realized'].sum()),
    'recorded_sales_amount_all_analysis_rows': float(df['recorded_sales_amount'].sum()),
    'net_sales': float(df['net_sales'].sum()),
    'net_profit': float(df['net_profit'].sum()),
    'sales_formula_mismatches_gt_0_02': int((sales_diff > 0.02).sum()),
    'profit_formula_mismatches_gt_0_02': int((profit_diff > 0.02).sum()),
    'dictionary_available': bool(SALES_DICT_PATH.exists()),
    'notes': [
        'Dictionary defines Sales_Amount as recorded net sales and Profit as recorded transaction profit.',
        'Dictionary describes Order_Status as operational status; Completed-only recognition remains an explicit analyst assumption.',
        'Missing customer_age and customer_rating are retained; no row deletion or median imputation.',
        'B2B Portal is separated from bulk-order quantity; other channels are not automatically labelled B2C.',
        'Financial formula discrepancies are reported, not silently overwritten.'
    ]
}
AUDIT_PATH.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding='utf-8')

# Post-save checks
sales_check = pd.read_csv(SALES_BI_READY_PATH)
rfm_check = pd.read_csv(RFM_BI_READY_PATH)
assert sales_check['transaction_id'].notna().all()
assert {'is_realized','net_sales','net_profit','is_b2b_channel','is_bulk_order_line'}.issubset(sales_check.columns)
assert rfm_check['customer_id'].is_unique
assert rfm_check['segment'].notna().all()
assert np.isclose(sales_check['net_sales'].sum(), df['net_sales'].sum())

print('Сохранено:')
for p in [SALES_CLEAN_PATH, SALES_BI_READY_PATH, RFM_BI_READY_PATH, AUDIT_PATH, ANOMALIES_PATH]:
    print(f' - {p}')
print('Проверки после сохранения пройдены.')


Сохранено:
 - /Users/caseynger/DE/final_work_fa/processed/sales_clean.csv
 - /Users/caseynger/DE/final_work_fa/processed/sales_bi_ready.csv
 - /Users/caseynger/DE/final_work_fa/processed/rfm_bi_ready.csv
 - /Users/caseynger/DE/final_work_fa/processed/etl_audit.json
 - /Users/caseynger/DE/final_work_fa/raw/quantity_anomalies.csv
Проверки после сохранения пройдены.
